# Laya GPU server (for laya-vs-jev-pong)

Runs the same `laya` model this repo's local `server/laya_server.py` runs, but on a real GPU via Colab's free Tesla T4 - matching the ~40ms figure in Laya's own docs, instead of the ~1s+ this checkpoint takes on a typical CPU-only dev machine.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.

You'll also need a free [ngrok](https://dashboard.ngrok.com/signup) account - Colab has no public IP of its own, so ngrok tunnels this notebook's local port 8787 out to a real HTTPS URL your browser can reach. Grab your authtoken from https://dashboard.ngrok.com/get-started/your-authtoken and paste it into cell 3 below.

In [ ]:
!pip install -q fastapi "uvicorn[standard]" laya pyngrok

import torch
assert torch.cuda.is_available(), "No GPU detected - check Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%%writefile colab_laya_server.py
# Same shape as this repo's server/laya_server.py, adapted to run
# standalone in a Colab VM: binds 0.0.0.0 (ngrok tunnels into it), forces
# device="cuda" instead of auto-detecting, and CORS defaults to the local
# Vite dev server's origin (override LAYA_CORS_ORIGIN if yours differs).
import os
import threading
from collections.abc import AsyncIterator
from contextlib import asynccontextmanager
from typing import Any

import laya
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

CHECKPOINT = "convaiinnovations/laya"
CHECKPOINT_SUBFOLDER = "typed-decisions"
_agent = None
_predict_lock = threading.Lock()


@asynccontextmanager
async def lifespan(_: FastAPI) -> AsyncIterator[None]:
    global _agent
    _agent = laya.load(CHECKPOINT, subfolder=CHECKPOINT_SUBFOLDER, device="cuda")
    yield


app = FastAPI(title="laya-vs-jev-pong: Colab GPU server", lifespan=lifespan)

_allowed_origin = os.environ.get("LAYA_CORS_ORIGIN", "http://localhost:5173")
app.add_middleware(
    CORSMiddleware,
    allow_origins=[_allowed_origin],
    allow_methods=["GET", "POST"],
    allow_headers=["content-type", "ngrok-skip-browser-warning"],
)


class DecideRequest(BaseModel):
    state: dict[str, Any]
    questions: dict[str, Any]


@app.get("/health")
def health() -> dict[str, Any]:
    return {"status": "ok" if _agent is not None else "loading", "checkpoint": CHECKPOINT_SUBFOLDER}


@app.post("/decide")
def decide(req: DecideRequest) -> dict[str, Any]:
    if _agent is None:
        raise HTTPException(status_code=503, detail="model not loaded yet")
    if not _predict_lock.acquire(blocking=False):
        raise HTTPException(status_code=503, detail="a decide request is already in flight")
    try:
        result = _agent.predict(req.state, req.questions)
    finally:
        _predict_lock.release()
    return {"model": CHECKPOINT_SUBFOLDER, "answers": result["answers"]}

In [ ]:
NGROK_AUTHTOKEN = ""  # paste yours from https://dashboard.ngrok.com/get-started/your-authtoken
# If your local Vite dev server isn't on the default port, change this to match:
import os
os.environ["LAYA_CORS_ORIGIN"] = "http://localhost:5173"

assert NGROK_AUTHTOKEN, "Set NGROK_AUTHTOKEN above first"

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [ ]:
import subprocess
import time

proc = subprocess.Popen(
    ["uvicorn", "colab_laya_server:app", "--host", "0.0.0.0", "--port", "8787"]
)
time.sleep(8)  # let the model finish loading onto the GPU before opening the tunnel

public_url = ngrok.connect(8787, "http")
print("Laya GPU server is live at:", public_url)
print()
print("Put this in your local .env:")
print(f"VITE_LAYA_BASE_URL={public_url}")
print()
print("Then restart `npm run dev:web` (Vite only re-reads .env on startup).")
print("You no longer need the local server/laya_server.py running at all.")

## Notes

- **Keep this tab open.** The server (and the tunnel) only exist while this notebook cell is running. Closing the tab or letting Colab idle-disconnect kills it.
- **Free ngrok URLs change every time** you rerun the tunnel cell - update `VITE_LAYA_BASE_URL` again if you restart.
- **Colab free tier GPUs aren't guaranteed** and sessions can be reclaimed after a few hours of use; this is a demo/dev convenience, not something to depend on for a long-running session.
- To stop the server cleanly: `proc.terminate()` in a new cell, then `ngrok.disconnect(public_url)`.